In [1]:
!pip install gpytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.6/280.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.7 MB/s eta 0:00:00


In [132]:
import torch
import gpytorch
import matplotlib.pyplot as plt
import numpy as np
import math


In [133]:
torch.manual_seed(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

In [134]:
# example with d = 3 and K = 3

# heterotopic data
Ns = [30, 20, 25]
K = 3
d = 3

def make_task_inputs(N, d=3):
    return (torch.rand(N, d, device=device, dtype=dtype) - 1.0)

def q1_and_grad(X):
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    val = 0.5 * (x1**2 + x2**2) + 0.1 * x3
    grad = torch.stack([x1, x2, 0.1 * torch.ones_like(x3)], dim=1)
    return val, grad

def q2_and_grad(X):
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    val = 0.5 * ((x1 - 0.5)**2 + 2.0 * (x2 + 0.2)**2) + 0.05 * (x3**2)
    grad = torch.stack([(x1 - 0.5), 2.0 * (x2 + 0.2), 0.1 * x3], dim=1)
    return val, grad

In [137]:
mix = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.6, 0.6],
], device=device, dtype=dtype)

val_noise_std  = torch.tensor([0.01, 0.01, 0.01], device=device, dtype=dtype)
grad_noise_std = torch.tensor([0.02, 0.02, 0.02], device=device, dtype=dtype)


In [138]:
# Build heterotopic per-task datasets
X_list, y_list, g_list = [], [], []
for t in range(K):
    X = make_task_inputs(Ns[t], d)
    q1, gq1 = q1_and_grad(X)
    q2, gq2 = q2_and_grad(X)

    a, b = mix[t, 0], mix[t, 1]
    f = a * q1 + b * q2
    g = a * gq1 + b * gq2

    y = f + val_noise_std[t] * torch.randn_like(f)
    g_noisy = g + grad_noise_std[t] * torch.randn_like(g)

    X_list.append(X)
    y_list.append(y)        # (Nt,)
    g_list.append(g_noisy)  # (Nt, d)

In [139]:
# ----------------------------
# Stack base points once (NO channel_id, NO pack)
# y_vec is interleaved per point: [y_i, g_i1, g_i2, g_i3, ...]
# ----------------------------
def build_stacked_dataset(X_list, y_list, g_list, val_noise_std, grad_noise_std):
    d = X_list[0].shape[1]
    device = X_list[0].device

    X_all = torch.cat(X_list, dim=0)  # (Ntot, d)

    task_id = torch.cat([
        torch.full((X.shape[0],), t, device=device, dtype=torch.long)
        for t, X in enumerate(X_list)
    ], dim=0)  # (Ntot,)

    # (Ntot, d+1)
    Y_mat = torch.cat([
        torch.cat([y.view(-1, 1), g.view(-1, d)], dim=1)
        for y, g in zip(y_list, g_list)
    ], dim=0)

    y_vec = Y_mat.reshape(-1)  # (Ntot*(d+1),)

    # noise per point then expanded to channels
    val_var  = (val_noise_std[task_id] ** 2)    # (Ntot,)
    grad_var = (grad_noise_std[task_id] ** 2)   # (Ntot,)

    noise_mat = torch.cat([val_var[:, None], grad_var[:, None].repeat(1, d)], dim=1)  # (Ntot, d+1)
    noise_vec = noise_mat.reshape(-1)  # (Ntot*(d+1),)

    # X_aug = [x..., task_id]
    X_aug = torch.cat([X_all, task_id.float().unsqueeze(1)], dim=1)  # (Ntot, d+1)

    return X_aug, y_vec, noise_vec

X_aug, y_vec, noise_vec = build_stacked_dataset(X_list, y_list, g_list, val_noise_std, grad_noise_std)

Ntot = X_aug.size(0)
assert y_vec.numel() == Ntot * (d + 1)
assert noise_vec.numel() == y_vec.numel()
print("Train base points Ntot =", Ntot, "AND", "y_vec length =", y_vec.numel())

Train base points Ntot = 75 AND y_vec length = 300


In [140]:
X_aug, y_vec, noise_vec = build_stacked_dataset(
    X_list, y_list, g_list, val_noise_std, grad_noise_std
)

Ntot = X_aug.size(0)
print("Ntot base points:", Ntot)
print("X_aug shape:", X_aug.shape)           # (Ntot, d+1)
print("y_vec shape:", y_vec.shape)           # (Ntot*(d+1),)
print("noise_vec shape:", noise_vec.shape)   # (Ntot*(d+1),)

Ntot base points: 75
X_aug shape: torch.Size([75, 4])
y_vec shape: torch.Size([300])
noise_vec shape: torch.Size([300])


In [141]:
class LMCGradKernel(gpytorch.kernels.Kernel):
    def __init__(self, num_tasks: int, input_dim: int, num_latents: int = 2, rank: int = 1):
        super().__init__()
        self.d = input_dim
        self.Q = num_latents

        self.task_kernels = torch.nn.ModuleList([
            gpytorch.kernels.IndexKernel(num_tasks=num_tasks, rank=rank)
            for _ in range(self.Q)
        ])
        self.grad_kernels = torch.nn.ModuleList([
            gpytorch.kernels.RBFKernelGrad(ard_num_dims=input_dim)
            for _ in range(self.Q)
        ])

    def num_outputs_per_input(self, x1=None, x2=None):
        return self.d + 1

    def forward(self, x1, x2, **params):
        d = self.d
        x1_data = x1[:, :d]
        x2_data = x2[:, :d]
        t1 = x1[:, d].long()
        t2 = x2[:, d].long()

        K = None
        for q in range(self.Q):
            # (n1*(d+1), n2*(d+1))
            Kx = self.grad_kernels[q](x1_data, x2_data, **params).to_dense()

            # (n1, n2) -> expand
            Kt = self.task_kernels[q](t1, t2).to_dense()
            Kt_big = Kt.repeat_interleave(d + 1, dim=0).repeat_interleave(d + 1, dim=1)

            Kq = Kx * Kt_big
            K = Kq if K is None else (K + Kq)

        return K

In [142]:
class ExactLMCGradGP(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks, input_dim, num_latents=2, rank=1):
        super().__init__(train_x, train_y, likelihood)
        self.d = input_dim

        self.mean_module = gpytorch.means.ConstantMeanGrad()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            LMCGradKernel(num_tasks=num_tasks, input_dim=input_dim, num_latents=num_latents, rank=rank)
        )

    def forward(self, x_aug):
        x_data = x_aug[:, :self.d]
        mean_vec = self.mean_module(x_data).reshape(-1)     # (N*(d+1),)
        cov = self.covar_module(x_aug, x_aug)               # (N*(d+1), N*(d+1))
        return gpytorch.distributions.MultivariateNormal(mean_vec, cov)

likelihood = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=noise_vec).to(device)
model = ExactLMCGradGP(X_aug, y_vec, likelihood, num_tasks=K, input_dim=d, num_latents=2, rank=1).to(device)

In [143]:
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

iters = 200
for it in range(iters):
    optimizer.zero_grad()
    out = model(X_aug)
    loss = -mll(out, y_vec)
    loss.backward()
    optimizer.step()

    if (it + 1) % 10 == 0:
        print(f"Iter {it+1}/{iters} | loss = {loss.item():.4f}")

Iter 10/200 | loss = -1.3599
Iter 20/200 | loss = -1.4864
Iter 30/200 | loss = -1.5863
Iter 40/200 | loss = -1.6657
Iter 50/200 | loss = -1.7307
Iter 60/200 | loss = -1.7841
Iter 70/200 | loss = -1.8303
Iter 80/200 | loss = -1.8696
Iter 90/200 | loss = -1.9037
Iter 100/200 | loss = -1.9334
Iter 110/200 | loss = -1.9593
Iter 120/200 | loss = -1.9819
Iter 130/200 | loss = -2.0020
Iter 140/200 | loss = -2.0197
Iter 150/200 | loss = -2.0357
Iter 160/200 | loss = -2.0496
Iter 170/200 | loss = -2.0624
Iter 180/200 | loss = -2.0736
Iter 190/200 | loss = -2.0839
Iter 200/200 | loss = -2.0936


In [144]:
def rmse(a, b):
    return torch.sqrt(torch.mean((a - b) ** 2)).item()

def task_truth(t, X):
    q1, g1 = q1_and_grad(X)
    q2, g2 = q2_and_grad(X)
    a, b = mix[t, 0], mix[t, 1]
    return a*q1 + b*q2, a*g1 + b*g2

model.eval()
likelihood.eval()

Ntest = 200
rmse_vals = []
rmse_grads = []

In [145]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    for t in range(K):
        X_test = make_task_inputs(Ntest, d)
        X_test_aug = torch.cat([X_test, torch.full((Ntest, 1), float(t), device=device, dtype=dtype)], dim=1)

        post = model(X_test_aug)  # latent posterior over [f, grads] flattened
        mean_mat = post.mean.view(Ntest, d + 1)

        f_pred = mean_mat[:, 0]
        g_pred = mean_mat[:, 1:]  # (Ntest, d)

        f_true, g_true = task_truth(t, X_test)

        r_f = rmse(f_pred, f_true)
        r_g = rmse(g_pred, g_true)

        rmse_vals.append(r_f)
        rmse_grads.append(r_g)

        print(f"[Task {t}] RMSE(f) = {r_f:.6f} | RMSE(grad) = {r_g:.6f}")

print("\nAverage RMSE(f)   =", sum(rmse_vals) / K)
print("Average RMSE(grad)=", sum(rmse_grads) / K)

[Task 0] RMSE(f) = 0.002923 | RMSE(grad) = 0.012709
[Task 1] RMSE(f) = 0.006308 | RMSE(grad) = 0.021970
[Task 2] RMSE(f) = 0.007200 | RMSE(grad) = 0.027805

Average RMSE(f)   = 0.005477044498547912
Average RMSE(grad)= 0.020828014550109703


/usr/local/lib/python3.12/dist-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-05 to the diagonal
  warnings.warn(


In [146]:
t_star = 2
X_query = make_task_inputs(5, d)
X_query_aug = torch.cat([X_query, torch.full((X_query.size(0), 1), float(t_star), device=device, dtype=dtype)], dim=1)

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    post = model(X_query_aug)
    mean_mat = post.mean.view(X_query.size(0), d + 1)

print("\nExample predictions for task", t_star)
print("f_mean:", mean_mat[:, 0])
print("grad_mean:", mean_mat[:, 1:])


Example predictions for task 2
f_mean: tensor([0.2786, 0.4682, 0.7401, 0.3711, 0.3067])
grad_mean: tensor([[-0.6055, -0.6007,  0.0310],
        [-1.0113, -0.3400,  0.0273],
        [-0.4818, -1.4529,  0.0141],
        [-0.7156, -0.6222,  0.0431],
        [-0.6261, -0.6023,  0.0335]])


In [147]:
def task_truth(t, X):
    q1, g1 = q1_and_grad(X)
    q2, g2 = q2_and_grad(X)
    a, b = mix[t, 0], mix[t, 1]
    return a*q1 + b*q2, a*g1 + b*g2

def rmse(a, b):
    return torch.sqrt(torch.mean((a - b) ** 2)).item()

model.eval()
likelihood.eval()

Ntest = 200

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    for t in range(K):
        X_test = make_task_inputs(Ntest, d)

        # build augmented test input with task id
        X_test_aug = torch.cat(
            [X_test, torch.full((Ntest, 1), float(t), device=device, dtype=dtype)],
            dim=1
        )

        # latent posterior over [f, grads] (noise-free)
        post = model(X_test_aug)
        mean_mat = post.mean.view(Ntest, d + 1)

        f_pred = mean_mat[:, 0]
        g_pred = mean_mat[:, 1:]

        f_true, g_true = task_truth(t, X_test)

        print(f"[Task {t}] RMSE(f_true)   = {rmse(f_pred, f_true):.6f}")
        print(f"[Task {t}] RMSE(grad_true)= {rmse(g_pred, g_true):.6f}\n")

[Task 0] RMSE(f_true)   = 0.003595
[Task 0] RMSE(grad_true)= 0.015059

[Task 1] RMSE(f_true)   = 0.005481
[Task 1] RMSE(grad_true)= 0.021691

[Task 2] RMSE(f_true)   = 0.004580
[Task 2] RMSE(grad_true)= 0.019694

